In [18]:
import pandas as pd 
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

data= pd.read_csv("upi_transactions.csv")

print("Shape:",data.shape)
print("Fraud Detection",data["fraud"].value_counts())

Shape: (50000, 12)
Fraud Detection fraud
0    46937
1     3063
Name: count, dtype: int64


In [19]:
# Split feature and Target
X=data.drop("fraud",axis=1)
y=data["fraud"] 

#identifying categorical and numerical columns
categorical_cols=["transaction_type","merchant_category","device_type"]
numeric_cols=[col for col in X.columns if col not in categorical_cols]

#Preprocessing pipeline

preprocessor=ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),numeric_cols),
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_cols)
    ]
)

#fit and transform

X_processed=preprocessor.fit_transform(X)

print("Processed Shape:",X_processed.shape)

Processed Shape: (50000, 19)


In [20]:
#Handle imbalanced data using SMOTE

smote =SMOTE(sampling_strategy=0.4,random_state=42)
X_resampled,y_resampled=smote.fit_resample(X_processed,y)

print("Shape of Features:",X_resampled.shape)
print("Count of fraud:",y_resampled.value_counts())

Shape of Features: (65711, 19)
Count of fraud: fraud
0    46937
1    18774
Name: count, dtype: int64


In [21]:
#train_test_split

X_train,X_test,y_train,y_test=train_test_split(X_resampled,y_resampled,test_size=0.25,random_state=42)

print("Train data shape:",X_train.shape)
print("Test data shape:",X_test.shape)

Train data shape: (49283, 19)
Test data shape: (16428, 19)


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score,recall_score,precision_score,confusion_matrix,f1_score,roc_auc_score,classification_report)


models={
    "logisticRegression":LogisticRegression(max_iter=500),
    "Random Forest":RandomForestClassifier(n_estimators=200,
                                           max_depth=12,
                                           class_weight="balanced",
                                           random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.08,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
}

results=[]

In [23]:
#train and evaluate models

for name,model in models.items():
    print(f"Training:{name}")
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    y_proba=model.predict_proba(X_test)[:,1]
    
    
    metrics={
        "Model":name,
        "Accuracy":accuracy_score(y_test,y_pred),
        "Precision":precision_score(y_test,y_pred),
        "Recall":recall_score(y_test,y_pred),
        "F1 score":f1_score(y_test,y_pred),
        "ROC AUC":roc_auc_score(y_test,y_proba)
    }
    
    results.append(metrics)
    
    print("classification_report:",classification_report(y_test,y_pred))
    print("confusion_matrix:",confusion_matrix(y_test,y_pred))
    
    
result_df=pd.DataFrame(results)
print("Model Performance Comparisons.")
print(result_df)

Training:logisticRegression
classification_report:               precision    recall  f1-score   support

           0       0.94      1.00      0.97     11680
           1       1.00      0.83      0.91      4748

    accuracy                           0.95     16428
   macro avg       0.97      0.92      0.94     16428
weighted avg       0.95      0.95      0.95     16428

confusion_matrix: [[11680     0]
 [  790  3958]]
Training:Random Forest
classification_report:               precision    recall  f1-score   support

           0       0.98      1.00      0.99     11680
           1       1.00      0.94      0.97      4748

    accuracy                           0.98     16428
   macro avg       0.99      0.97      0.98     16428
weighted avg       0.98      0.98      0.98     16428

confusion_matrix: [[11680     0]
 [  292  4456]]
Training:XGBoost
classification_report:               precision    recall  f1-score   support

           0       0.99      1.00      0.99     11680
  

In [24]:
import pickle

# Save the best model (XGBoost)
best_model = models["XGBoost"]

with open("fraud_model_xgb.pkl", "wb") as f:
    pickle.dump(best_model, f)

# Save the preprocessing pipeline
with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

print("✅ Model & Preprocessor saved successfully!")



✅ Model & Preprocessor saved successfully!
